# 2026-10-16_1 scTenifoldKnk 新一轮 9 TF stage 拆分 virtual KO（MG）

新增 9 个候选 TF 按stage 拆分运行（young / old × central / peripheral，每组合独立建网并 KO）：KLF9、ZNF460、ZNF148、MAFK、NFE2L1、KLF15、THRB、NR2F1（均 MG）与 **E2F6（Cone，经用户更正）**。

- 阈值口径（经用户决定，混合策略）：默认官方 **minPCT = 0.05**；预检中 QC 后检出率 <5% 的 6 个组合放宽至 **0.01**（ZNF460 young/old-central 与 young-peripheral、NFE2L1 young-peripheral、KLF15 old-central 与 old-peripheral）；E2F6 在 Cone 中 4 组均通过 0.05（5.2–9.8%），无需放宽
- THRB(var.3)：上游 motif 变异标注，实际 KO 基因为 **THRB**
- E2F6 的 target ZIC1 在 Cone-central 检出率 0.19–0.39%，任何阈值均不可行，该侧 target 将记 NA（peripheral 两侧正常）
- 注意：0.01 网络与 0.05 网络不可直接比较（gene universe 差异大）；同一 TF 各 stage 网络阈值不同时 stage 间比较需注意口径；低检出基因的 GRN 边可靠性弱于官方口径

其余规则与参数不变：官方 scQC（minLibSize=1000, removeOutlierCells=TRUE, maxMTratio=0.1）+ 官方 ribo/mito regex，`qc = FALSE`，nc_nNet=10 / nc_nCells=500 / td_K=3，每组合 `set.seed(1)`。Stage 口径：young = 21/25/30/32years，old = 67/75/86years，70years 排除。

In [ ]:
# ---- 1. 环境和参数 ------------------------------------------------------------
# 已测试 virtual KO（pooled，0.05）: MEF2C(MG) NFIC(MG) RORB(MG) ZIC1(Cone，仅 peripheral)
# 已测试 virtual KO（stage 拆分，0.05）: MEF2C/NFIC/RORB(MG) ZIC1(Cone-p)
# 已测试 virtual KO（stage 拆分，0.01/0.009 放宽）: EGR1(BC) ONECUT1(MG) USF1(MG)
# ZIC1(Cone)-central 检出率 0.19-0.39%，任何阈值均不可行
# 本脚本（stage 拆分，混合阈值 0.05/0.01）: KLF9 ZNF460 ZNF148 MAFK NFE2L1 KLF15 THRB NR2F1（MG）+ E2F6（Cone，经用户更正）
#   THRB(var.3) 为上游标注，KO 基因为 THRB
#   0.01 组合: ZNF460(yc/oc/yp) NFE2L1(yp) KLF15(oc/op)；E2F6 在 Cone 全组通过 0.05，无需放宽
suppressPackageStartupMessages({
  library(Seurat)
  library(Matrix)
  library(data.table)
  library(scTenifoldKnk)
})
set.seed(1)

# 所有可修改参数集中在此
INPUT_RDS <- "/project/work/2026-05-25_create_summary/Data/2026-06-05_new_adult_human_harmony_anno.rds"
OUT_DIR   <- "/project/work/2026-10-13_scTenifoldKnk/Data/"
DATE      <- "2026-10-16"

NC_NNET   <- 10    # nc_nNet
NC_NCELLS <- 500   # nc_nCells，官方默认
TD_K      <- 3     # td_K
NCORES    <- 10    # 并行核数

# stage 定义（沿用项目口径；70years 为 Old_unpaired，无 peripheral 对照，排除）
YOUNG_AGES <- c("21years", "25years", "30years", "32years")
OLD_AGES   <- c("67years", "75years", "86years")

# QC 基因过滤阈值（混合策略，经用户决定）：默认官方 0.05，预检 <5% 的组合放宽 0.01
MINPCT_DEFAULT  <- 0.05
MINPCT_RELAXED  <- 0.01

KO_PLAN <- data.table(
  TF       = rep(c("KLF9", "ZNF460", "ZNF148", "MAFK", "NFE2L1", "KLF15", "E2F6", "THRB", "NR2F1"), each = 4),
  celltype = rep(c("MG", "MG", "MG", "MG", "MG", "MG", "Cone", "MG", "MG"), each = 4),
  region   = rep(c("central", "peripheral"), each = 2),
  stage    = rep(c("young", "old"), 18)
)
KO_PLAN[, minpct := MINPCT_DEFAULT]
KO_PLAN[TF == "ZNF460" & !(region == "peripheral" & stage == "old"), minpct := MINPCT_RELAXED]
KO_PLAN[TF == "NFE2L1" & region == "peripheral" & stage == "young", minpct := MINPCT_RELAXED]
KO_PLAN[TF == "KLF15" & stage == "old", minpct := MINPCT_RELAXED]
print(KO_PLAN)

dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)

# 确认服务器实际安装版本（不安装/升级任何 package）
packageVersion("scTenifoldKnk")
packageVersion("scTenifoldNet")
args(scTenifoldKnk)

In [ ]:
# ---- 2. 读取 Seurat object，统计各组合细胞数与目标基因 raw 检出率 --------------
obj <- readRDS(INPUT_RDS)
cat("总细胞数:", ncol(obj), "\n")

# Seurat v5：RNA assay 仅存在唯一标准 counts layer（沿用 pilot 逻辑）
print(SeuratObject::Layers(obj[["RNA"]]))
if (!"counts" %in% SeuratObject::Layers(obj[["RNA"]])) {
  stop("RNA assay 中不存在 counts layer")
}
counts_raw <- SeuratObject::LayerData(obj, assay = "RNA", layer = "counts")
stopifnot(nrow(counts_raw) > 0, ncol(counts_raw) > 0)
stopifnot(identical(colnames(counts_raw), colnames(obj)))

stopifnot(all(c("celltype", "tissue", "donor_age") %in% colnames(CONTACT_EMAIL)))
meta_dt <- data.table(cell = colnames(obj), celltype = obj$celltype,
                      tissue = obj$tissue, donor_age = obj$donor_age)
rm(obj); gc()

# stage 映射；无 stage 归属的细胞（70years，Old_unpaired）排除并记录
meta_dt[, stage := fcase(donor_age %in% YOUNG_AGES, "young",
                         donor_age %in% OLD_AGES, "old",
                         default = NA_character_)]
cat("排除的细胞（无 stage 归属）:", meta_dt[is.na(stage), .N], "\n")

# 项目既有口径：fovea + macula -> central，peripheral -> peripheral
qc_raw <- rbindlist(lapply(seq_len(nrow(KO_PLAN)), function(i) {
  tp <- KO_PLAN[i]
  tissues <- if (tp$region == "central") c("fovea", "macula") else "peripheral"
  cells <- meta_dt[celltype == tp$celltype & tissue %in% tissues & stage == tp$stage, cell]
  v <- counts_raw[tp$TF, cells]
  data.table(TF = tp$TF, celltype = tp$celltype, region = tp$region, stage = tp$stage,
             minpct = tp$minpct,
             raw_cells = length(cells),
             gene_detected_cells = sum(v > 0),
             gene_detection_pct = round(100 * mean(v > 0), 2))
}))
print(qc_raw)

# 目标基因必须存在于 rownames（不允许静默删除）
if (!all(KO_PLAN$TF %in% rownames(counts_raw))) {
  stop("以下目标基因不在 RNA counts 的 rownames 中: ",
       paste(setdiff(KO_PLAN$TF, rownames(counts_raw)), collapse = ", "))
}
cat("raw cells < 500 的组合数:", qc_raw[raw_cells < NC_NCELLS, .N], "\n")

In [ ]:
# ---- 3. 逐组合 scQC + ribo/mito 过滤（独立 gene universe，逐组合阈值）----------
# 同一 (region, stage, minpct) 的 QC 结果相同，用缓存避免重复计算
X_list <- list(); qc_cache <- list()
qc_dt <- copy(qc_raw)
qc_dt[, `:=`(qc_cells = NA_integer_, qc_genes = NA_integer_, gene_passed_QC = NA)]

for (i in seq_len(nrow(KO_PLAN))) {
  tp <- KO_PLAN[i]
  ck <- paste(tp$celltype, tp$region, tp$stage, tp$minpct, sep = "_")
  if (is.null(qc_cache[[ck]])) {
    tissues <- if (tp$region == "central") c("fovea", "macula") else "peripheral"
    cells <- meta_dt[celltype == tp$celltype & tissue %in% tissues & stage == tp$stage, cell]
    X <- scQC(counts_raw[, cells], minLibSize = 1000, removeOutlierCells = TRUE,
              minPCT = tp$minpct, maxMTratio = 0.1)
    riboMito <- grepl("^(RP[LS][0-9]|RPLP[0-9]|RPSA|MRP[LS][0-9]|MT-|MTND[0-9]|MTCO[0-9]|MTATP[0-9]|MTCYB|MTRNR2L)",
                      toupper(rownames(X)))
    qc_cache[[ck]] <- X[!riboMito, ]
  }
  X <- qc_cache[[ck]]
  X_list[[paste(tp$TF, tp$celltype, tp$region, tp$stage, sep = "_")]] <- X
  idx <- which(qc_dt$TF == tp$TF & qc_dt$celltype == tp$celltype &
               qc_dt$region == tp$region & qc_dt$stage == tp$stage)
  set(qc_dt, idx, "qc_cells", ncol(X))
  set(qc_dt, idx, "qc_genes", nrow(X))
  set(qc_dt, idx, "gene_passed_QC", tp$TF %in% rownames(X))
  cat(sprintf("%s (%s, %s, %s, minPCT=%.2f): QC 后 %d cells x %d genes | 目标基因保留: %s\n",
              tp$TF, tp$celltype, tp$region, tp$stage, tp$minpct, ncol(X), nrow(X), tp$TF %in% rownames(X)))
}
rm(counts_raw, qc_cache); gc()

# QC 后判定：基因保留且 cells >= 500 才运行；否则跳过并记录原因（不降阈值、不自动调 nc_nCells）
qc_dt[, run := gene_passed_QC & qc_cells >= NC_NCELLS]
qc_dt[, skip_reason := fcase(
  !gene_passed_QC, paste0("target gene filtered by scQC (minPCT = ", minpct, ")"),
  qc_cells < NC_NCELLS, paste0("qc_cells < nc_nCells (", NC_NCELLS, ")"),
  default = "")]
print(qc_dt)

In [ ]:
# ---- 4. scTenifoldKnk virtual KO（仅运行通过 QC 的组合）------------------------
run_dt <- qc_dt[run == TRUE]
run_keys <- paste(run_dt$TF, run_dt$celltype, run_dt$region, run_dt$stage, sep = "_")
for (k in setdiff(names(X_list), run_keys)) X_list[[k]] <- NULL  # 释放不运行组合的矩阵

cat("将运行的 KO 组合数:", nrow(run_dt), "\n")
print(run_dt[, .(TF, celltype, region, stage, minpct, qc_cells, qc_genes)])

ko_out <- list()
runtime_dt <- data.table()
for (i in seq_len(nrow(run_dt))) {
  tf <- run_dt$TF[i]; ct <- run_dt$celltype[i]; rg <- run_dt$region[i]; st <- run_dt$stage[i]
  key <- paste(tf, ct, rg, st, sep = "_")
  cat(sprintf("\n=== [%d/%d] %s (%s, %s, %s) %d genes x %d cells ===\n",
              i, nrow(run_dt), tf, ct, rg, st, run_dt$qc_genes[i], run_dt$qc_cells[i]))
  set.seed(1)  # 每个组合同一 seed，保证随机 subsampling 可复现
  tm <- system.time(
    ko_out[[key]] <- scTenifoldKnk(
      countMatrix = X_list[[key]],
      gKO = tf,
      qc = FALSE,
      nc_nNet = NC_NNET,
      nc_nCells = NC_NCELLS,
      td_K = TD_K,
      nCores = NCORES
    )
  )
  runtime_dt <- rbind(runtime_dt, data.table(
    TF = tf, celltype = ct, region = rg, stage = st,
    elapsed = tm[["elapsed"]], user = tm[["user.self"]], system = tm[["sys.self"]]))
  dr <- ko_out[[key]]$diffRegulation
  cat(sprintf("耗时 %.1f min | diffRegulation %d genes | p.adj < 0.05: %d\n",
              tm[["elapsed"]] / 60, nrow(dr), sum(dr$p.adj < 0.05, na.rm = TRUE)))
}
print(runtime_dt)

In [ ]:
# ---- 5. 保存输出 --------------------------------------------------------------
for (i in seq_len(nrow(run_dt))) {
  tf <- run_dt$TF[i]; ct <- run_dt$celltype[i]; rg <- run_dt$region[i]; st <- run_dt$stage[i]
  key <- paste(tf, ct, rg, st, sep = "_")
  base <- paste0(DATE, "_", ct, "_", rg, "_", st, "_", tf)
  saveRDS(ko_out[[key]], file.path(OUT_DIR, paste0(base, "_scTenifoldKnk.rds")))
  fwrite(ko_out[[key]]$diffRegulation, file.path(OUT_DIR, paste0(base, "_diffRegulation.tsv")), sep = "\t")
  cat("已保存:", base, "\n")
}

fwrite(qc_dt, file.path(OUT_DIR, paste0(DATE, "_newTF9_stage_split_KO_QC.tsv")), sep = "\t")
fwrite(runtime_dt, file.path(OUT_DIR, paste0(DATE, "_newTF9_stage_split_KO_runtime.tsv")), sep = "\t")

cat("\n=== QC 汇总 ===\n"); print(qc_dt)
cat("\n=== runtime 汇总 ===\n"); print(runtime_dt)
cat("\n输出文件:\n")
print(sort(list.files(OUT_DIR, pattern = paste0("_(KLF9|ZNF460|ZNF148|MAFK|NFE2L1|KLF15|E2F6|THRB|NR2F1)_(scTenifoldKnk|diffRegulation)\\.|^", DATE, "_newTF9_"), full.names = TRUE)))